# Submission C: 1.000 < aloo/guest < 1.975

| | |
|---|---|
| Submitted file | `sub_C_band_1p00_1p975.csv` |
| Submitted (UTC) | 2026-10-07 16:35 |
| Public score | 0.95580 (173/181) |
| Rule | 1.000 < aloo/guest < 1.975 |

C shifts both edges (lower to 1.0, upper to 1.975). Every leaderboard team uses nearly the same band, so C is a ticket other teams do not hold (E21/E22, `solution/win_prob.py`, `solution/win_prob2.py`). The same predictions were submitted again at 17:45 UTC from notebook [kacchi-aloo-band-submission](https://www.kaggle.com/code/hosen42/kacchi-aloo-band-submission) v2.

**Run it:** on Kaggle, upload this notebook and add the competition data (*Add Input → Competitions → The Great Kacchi Aloo Mystery*), then *Save & Run All* and submit `submission.csv`. Locally, run it from this folder (it reads `../data/train.csv` and `../data/test.csv`). The last cell checks the output against the file that was actually submitted.

In [1]:
import glob, hashlib, os, unicodedata
import numpy as np, pandas as pd

found = sorted(glob.glob("/kaggle/input/**/train.csv", recursive=True)) + [p for p in ("../data/train.csv", "data/train.csv") if os.path.exists(p)]
root = os.path.dirname(found[0])
tr_raw = pd.read_csv(f"{root}/train.csv", dtype=str)
te_raw = pd.read_csv(f"{root}/test.csv", dtype=str)


def to_number(v):
    """Float from a cell; any Unicode decimal digit (e.g. Bangla) becomes ASCII."""
    if pd.isna(v):
        return np.nan
    s = "".join(str(unicodedata.decimal(c)) if unicodedata.decimal(c, None) is not None else c
                for c in str(v).strip().replace(",", ""))
    try:
        return float(s)
    except ValueError:
        return np.nan


def add_apg(d):
    """aloo per guest; the few aloo counts typed x10 (ratio > 3) are divided by 10."""
    guests, aloo = d.guests.map(to_number), d.aloo_count.map(to_number)
    aloo = np.where(aloo / guests > 3.0, aloo / 10, aloo)
    return d.assign(apg=aloo / guests)


tr = add_apg(tr_raw).dropna(subset=["apg"]).reset_index(drop=True)   # 24 train rows have no aloo_count
tr["y"] = tr.went_back_for_seconds.astype(int)
te = add_apg(te_raw)
print(f"data: {root} | train {len(tr)} rows with aloo_count | test {len(te)} rows")

data: ../data | train 776 rows with aloo_count | test 400 rows


In [2]:
LO, HI = 1.0, 1.975   # variant C: chosen by the final-selection simulation (solution/win_prob.py) as B's partner
inside = lambda x: ((x > LO) & (x < HI)).astype(int)
print(f"band {LO} < aloo/guest < {HI}; train accuracy {(inside(tr.apg.values) == tr.y.values).mean():.4f}")
pred = inside(te.apg.values)

band 1.0 < aloo/guest < 1.975; train accuracy 0.9420


In [3]:
sub = pd.DataFrame({"wedding_id": te.wedding_id, "went_back_for_seconds": np.asarray(pred).astype(int)})
sub.to_csv("submission.csv", index=False)
fp = hashlib.md5("".join(f"{i}{p}" for i, p in zip(sub.wedding_id, sub.went_back_for_seconds)).encode()).hexdigest()
print(f"wrote submission.csv: {sub.went_back_for_seconds.sum()} ones / {len(sub)} rows")
print(f"identical to the submitted file 'sub_C_band_1p00_1p975.csv': {fp == '676efefb8674c3a4ebb1bd2fb05e19d3'}")

wrote submission.csv: 172 ones / 400 rows
identical to the submitted file 'sub_C_band_1p00_1p975.csv': True
